# One Queue or Two

This notebook presents a case study from *Modeling and Simulation in Python*.  It explores a question related to queueing theory, which is the study of systems that involve waiting in lines, also known as "queues".

Suppose you are designing the checkout area for a new store.  There is room for two checkout counters and a waiting area for customers.  You can make two lines, one for each counter, or one line that serves both counters.

In theory, you might expect a single line to be better, but it has some practical drawbacks: in order to maintain a single line, you would have to install rope barriers, and customers might be put off by what seems to be a longer line, even if it moves faster.

So you'd like to check whether the single line is really better and by how much.  Simulation can help answer this question.

As we did in the bikeshare model, we'll assume that a customer is equally likely to arrive during any timestep.  I'll denote this probability using the Greek letter lambda, $\lambda$, or the variable name `lam`.  The value of $\lambda$ probably varies from day to day, so we'll have to consider a range of possibilities.

Based on data from other stores, you know that it takes 5 minutes for a customer to check out, on average.  But checkout times are highly variable: most customers take less than 5 minutes, but some take substantially more.  A simple way to model this variability is to assume that when a customer is checking out, they have the same probability of finishing up during each time step.  I'll denote this probability using the Greek letter mu, $\mu$, or the variable name `mu`.

If we choose $\mu=1/5$, the average number of time steps for each checkout will be 5 minutes, which is consistent with the data.

## One server, one queue

Write a function called `make_system` that takes `lam` and `mu` as parameters and returns a `System` object with variables `lam`, `mu`, and `duration`.  Set `duration`, which is the number of time steps to simulate, to 10 hours, expressed in minutes.

In [5]:
# download modsim.py if necessary

from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print('Downloaded ' + local)

download('https://github.com/AllenDowney/ModSimPy/raw/master/modsim.py')

In [6]:
from modsim import *

In [ ]:
def make_system(lam, mu):
    """Makes a System object for the one-server queueing model.

    lam: arrival rate (customers per minute)
    mu: service rate (customers per minute)

    returns: System object
    """
    system = System(lam=lam, mu=mu, duration=600)
    return system

Test this function by creating a `System` object with `lam=1/8` and `mu=1/5`.

In [ ]:
system = make_system(lam=1/8, mu=1/5)
print(system)

Write an update function that takes as parameters `x`, which is the total number of customers in the store, including the one checking out; `t`, which is the number of minutes that have elapsed in the simulation, and `system`, which is a `System` object.

If there's a customer checking out, it should use `flip` to decide whether they are done.  And it should use `flip` to decide if a new customer has arrived.

It should return the total number of customers at the end of the time step.

In [ ]:
def update_func(x, t, system):
    """Update the number of customers in the queueing system.

    x: total number of customers in the store
    t: current time step
    system: System object

    returns: updated number of customers
    """
    # A customer finishes checking out
    if x > 0 and flip(system.mu):
        x -= 1

    # A new customer arrives
    if flip(system.lam):
        x += 1

    return x

Test your function by calling it with `x=1`, `t=0`, and the `System` object you created.  If you run it a few times, you should see different results.

In [ ]:
print(update_func(1, 0, system))
print(update_func(1, 0, system))
print(update_func(1, 0, system))

Now we can run the simulation.  Here's a version of `run_simulation` that creates a `TimeSeries` with the total number of customers in the store, including the one checking out.

In [ ]:
def run_simulation(system, update_func):
    """Simulate a queueing system.

    system: System object
    update_func: function object
    """
    x = 0
    results = TimeSeries(name='Queue length')
    results[0] = x

    for t in linrange(0, system.duration):
        x = update_func(x, t, system)
        results[t+1] = x

    return results

Call `run_simulation` with your update function and plot the results.

In [ ]:
results = run_simulation(system, update_func)
results.plot()
decorate(title='One Server, One Queue', xlabel='Time (minutes)', ylabel='Number of customers')

After the simulation, we can compute `L`, which is the average number of customers in the system, and `W`, which is the average time customers spend in the store.  `L` and `W` are related by Little's Law:

$L = \lambda W$

Where $\lambda$ is the arrival rate.  Here's a function that computes them.

In [ ]:
def compute_metrics(results, system):
    """Compute average number of customers and wait time.

    results: TimeSeries of queue lengths
    system: System object

    returns: L, W
    """
    L = results.mean()
    W = L / system.lam
    return L, W

Call `compute_metrics` with the results from your simulation.

In [ ]:
L, W = compute_metrics(results, system)
print(f'L = {L:.2f}')
print(f'W = {W:.2f}')

### Parameter sweep

Since we don't know the actual value of $\lambda$, we can sweep through a range of possibilities, from 10% to 80% of the completion rate, $\mu$.  (If customers arrive faster than the completion rate, the queue grows without bound.  In that case the metrics `L` and `W` just depend on how long the store is open.)

Create an array of values for `lam`.

In [ ]:
mu = system.mu
lam_array = linrange(0.1 * mu, 0.8 * mu, 20)
print(lam_array)

Write a function that takes an array of values for `lam`, a single value for `mu`, and an update function.

For each value of `lam`, it should run a simulation, compute `L` and `W`, and store the value of `W` in a `SweepSeries`.

It should return the `SweepSeries`.

In [ ]:
def sweep_lam(lam_array, mu, update_func):
    """Sweeps a range of `lam` values and returns a SweepSeries of W.

    lam_array: array of values for `lam`
    mu: service rate (customers per minute)
    update_func: function object

    returns: SweepSeries
    """
    sweep_series = SweepSeries()
    for lam in lam_array:
        system = make_system(lam, mu)
        results = run_simulation(system, update_func)
        L, W = compute_metrics(results, system)
        sweep_series[lam] = W
    return sweep_series

Call your function to generate a `SweepSeries`, and plot it.

In [ ]:
sweep_results = sweep_lam(lam_array, mu, update_func)
sweep_results.plot(label='simulation')
decorate(title='One Server, One Queue', xlabel='Arrival rate (customers per minute)', ylabel='Average wait time (minutes)')

In [ ]:
# Solution goes here

If we imagine that this range of values represents arrival rates on different days, we can use the average value of `W`, for a range of values of `lam`, to compare different queueing strategies.

### Analysis

The model I chose for this system is a common model in queueing theory, in part because many of its properties can be derived analytically.

In particular, we can derive the average time in the store as a function of $\mu$ and $\lambda$:

$W = 1 / (\mu - \lambda)$

The following function plots the theoretical value of $W$ as a function of $\lambda$.

In [ ]:
def plot_W(lam_array, mu):
    """Plot the theoretical mean wait time.

    lam_array: array of values for `lam`
    mu: probability of finishing a checkout
    """
    W_array = 1 / (mu - lam_array)
    W_series = make_series(lam_array, W_array)
    W_series.plot(style='-', label='analysis')

Use this function to plot the theoretical results, then plot your simulation results again on the same graph.  How do they compare?

In [ ]:
plot_W(lam_array, mu)
sweep_results.plot(label='simulation')

## Multiple servers

Now let's try the other two queueing strategies:

1.  One queue with two checkout counters.
2.  Two queues, one for each counter.

The following figure shows the three scenarios:

![](https://github.com/AllenDowney/ModSim/raw/main/figs/queue.png)

Write an update function for one queue with two servers.

In [ ]:
def update_func_2server(x, t, system):
    """Update the number of customers in the queueing system with two servers.

    x: total number of customers in the store
    t: current time step
    system: System object

    returns: updated number of customers
    """
    # Server 1 finishes checking out
    if x > 0 and flip(system.mu):
        x -= 1

    # Server 2 finishes checking out
    if x > 0 and flip(system.mu):
        x -= 1

    # A new customer arrives
    if flip(system.lam):
        x += 1

    return x

Use this update function to simulate the system, plot the results, and print the metrics.

In [ ]:
results_2server = run_simulation(system, update_func_2server)
results_2server.plot()
decorate(title='Two Servers, One Queue', xlabel='Time (minutes)', ylabel='Number of customers')

L2, W2 = compute_metrics(results_2server, system)
print(f'L = {L2:.2f}')
print(f'W = {W2:.2f}')

Since we have two checkout counters now, we can consider values for $\lambda$ that exceed $\mu$.

Create a new array of values for `lam` from 10% to 160% of `mu`.

In [ ]:
lam_array_2server = linrange(0.1 * mu, 1.6 * mu, 20)
print(lam_array_2server)

Use your sweep function to simulate the two server, one queue scenario with a range of values for `lam`.

Plot the results and print the average value of `W` across all values of `lam`.

In [ ]:
sweep_results_2server = sweep_lam(lam_array_2server, mu, update_func_2server)
sweep_results_2server.plot(label='Two servers, one queue')
decorate(title='Two Servers, One Queue', xlabel='Arrival rate (customers per minute)', ylabel='Average wait time (minutes)')

print(f'Average W for two servers, one queue: {sweep_results_2server.mean():.2f}')

In [ ]:
# Solution goes here

## Multiple queues

To simulate the scenario with two separate queues, we need two state variables to keep track of customers in each queue.

Write an update function that takes `x1`, `x2`, `t`, and `system` as parameters and returns `x1` and `x2` as return values.  f you are not sure how to return more than one return value, see `compute_metrics`.

When a customer arrives, which queue do they join?

In [ ]:
def update_func_2queue(x1, x2, t, system):
    """Update the number of customers in the two separate queueing systems.

    x1: total number of customers in queue 1
    x2: total number of customers in queue 2
    t: current time step
    system: System object

    returns: updated number of customers in x1, x2
    """

    # Server 1 finishes checking out
    if x1 > 0 and flip(system.mu):
        x1 -= 1

    # Server 2 finishes checking out
    if x2 > 0 and flip(system.mu):
        x2 -= 1

    # A new customer arrives
    if flip(system.lam):
        if x1 < x2:
            x1 += 1
        else:
            x2 += 1

    return x1, x2

Write a version of `run_simulation` that works with this update function.

In [ ]:
def run_simulation_2queue(system, update_func):
    """Simulate a queueing system with two queues.

    system: System object
    update_func: function object that returns x1, x2
    """
    x1 = 0
    x2 = 0
    results = TimeSeries(name='Total queue length')
    results[0] = x1 + x2

    for t in linrange(0, system.duration):
        x1, x2 = update_func(x1, x2, t, system)
        results[t+1] = x1 + x2

    return results

Test your functions by running a simulation with a single value of `lam`.

In [ ]:
results_2queue = run_simulation_2queue(system, update_func_2queue)
results_2queue.plot()
decorate(title='Two Queues, Two Servers', xlabel='Time (minutes)', ylabel='Number of customers')

L3, W3 = compute_metrics(results_2queue, system)
print(f'L = {L3:.2f}')
print(f'W = {W3:.2f}')

Sweep a range of values for `lam`, plot the results, and print the average wait time across all values of `lam`.

How do the results compare to the scenario with two servers and one queue.

In [ ]:
sweep_results_2queue_separate = sweep_lam(lam_array_2server, mu, update_func_2queue)
sweep_results_2queue_separate.plot(label='Two separate queues, two servers')
decorate(title='Two Separate Queues, Two Servers', xlabel='Arrival rate (customers per minute)', ylabel='Average wait time (minutes)')

print(f'Average W for two separate queues, two servers: {sweep_results_2queue_separate.mean():.2f}')

In [ ]:
# Solution goes here

In [ ]:
# Solution goes here

*Modeling and Simulation in Python*

Copyright 2021 Allen Downey

License: [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)